# Phase 0 — your first CUDA kernel on a free GPU

Runs on **Google Colab's free T4** — no local NVIDIA GPU needed. This is the hands-on CUDA piece of the inference-engine project: you compile a real `.cu` kernel, prove it matches PyTorch, and time it.

**Before running:** Runtime → Change runtime type → Hardware accelerator → **T4 GPU**.

The kernel here is `vector_add` (`c[i] = a[i] + b[i]`) — the 'hello world' of CUDA. It teaches the two fundamentals every later kernel (matmul, softmax, rmsnorm) reuses: the **flat thread index** and the **bounds guard**.

In [ ]:
!pip install -q ninja   # required by torch.utils.cpp_extension to compile .cu
# 1. Confirm we actually have a GPU (and see which one).
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'Set Runtime -> Change runtime type -> T4 GPU'

## 2. Write the kernel

`%%writefile` drops the CUDA source to disk so we can compile it. **Try it yourself first:** the two lines that matter are marked `# <-- YOU`. Cover them, and fill in (a) the bounds guard + write inside the kernel, and (b) the launch config. Then reveal and compare — they're the whole lesson.

- flat index: `idx = blockIdx.x * blockDim.x + threadIdx.x`
- guard: `if (idx < n) c[idx] = a[idx] + b[idx];`
- blocks: `(n + threads - 1) / threads`  (integer ceil division)

In [ ]:
%%writefile vector_add.cu
#include <torch/extension.h>
#include <cuda_runtime.h>

__global__ void vector_add_kernel(const float* a, const float* b, float* c, int n) {
    // Each thread computes ONE output element; this flat index is the CUDA basic.
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {                       // <-- YOU: the guard stops the last,
        c[idx] = a[idx] + b[idx];        //          partly-empty block writing OOB
    }
}

torch::Tensor vector_add(torch::Tensor a, torch::Tensor b) {
    TORCH_CHECK(a.is_cuda() && b.is_cuda(), "inputs must be CUDA tensors");
    TORCH_CHECK(a.sizes() == b.sizes(), "shapes must match");
    auto c = torch::empty_like(a);
    int n = a.numel();

    const int threads = 256;                        // <-- YOU: threads per block
    const int blocks = (n + threads - 1) / threads; // <-- YOU: ceil(n/threads)
    vector_add_kernel<<<blocks, threads>>>(
        a.data_ptr<float>(), b.data_ptr<float>(), c.data_ptr<float>(), n);
    return c;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("vector_add", &vector_add, "A + B (CUDA)");
}


## 3. Compile it (JIT) and check correctness

`torch.utils.cpp_extension.load` invokes `nvcc` under the hood, builds a Python module, and caches it. Colab's T4 is `sm_75`, so we target that arch. Then we assert `torch.allclose` against PyTorch's own `a + b` — the same pattern the repo's `kernels/tests/` use.

In [ ]:
from torch.utils.cpp_extension import load

ext = load(name='vector_add_ext', sources=['vector_add.cu'],
           extra_cuda_cflags=['-arch=sm_75'], verbose=True)  # T4 = sm_75

a = torch.randn(1_000_000, device='cuda')
b = torch.randn(1_000_000, device='cuda')
out = ext.vector_add(a, b)
ref = a + b
print('max abs diff:', (out - ref).abs().max().item())
assert torch.allclose(out, ref, atol=1e-5), 'kernel diverges from PyTorch!'
print('PASS: your CUDA kernel matches PyTorch ✅')

## 4. Benchmark it vs PyTorch

Kernel launches are **async**, so wall-clock timing measures nothing. We time with **CUDA events** (recorded on the GPU stream) and `synchronize` before reading — exactly what `kernels/tests/common.py::cuda_time_ms` does. At this size you're memory-bandwidth-bound, so expect to land in the same ballpark as PyTorch (which is the point — you wrote a kernel that's competitive on a trivial op).

In [ ]:
def cuda_time_ms(fn, warmup=10, iters=100):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    times = []
    for _ in range(iters):
        s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        s.record(); fn(); e.record(); torch.cuda.synchronize()
        times.append(s.elapsed_time(e))
    times.sort()
    return times[len(times)//2]

mine = cuda_time_ms(lambda: ext.vector_add(a, b))
torch_t = cuda_time_ms(lambda: a + b)
gb = a.numel() * 4 * 3 / 1e9  # read a, read b, write c = 3 arrays * 4 bytes
print(f'my kernel : {mine*1e3:8.1f} us   ({gb/(mine/1e3):6.1f} GB/s)')
print(f'pytorch   : {torch_t*1e3:8.1f} us   ({gb/(torch_t/1e3):6.1f} GB/s)')

## What you just did

- Compiled a hand-written CUDA kernel from source and called it from Python.
- Learned the flat thread index and why the bounds guard is mandatory.
- Measured it correctly with CUDA events and hit memory-bandwidth parity with PyTorch.

**Next kernels** (same pattern, more interesting): `softmax` (needs a per-row max + sum reduction — introduces shared memory) and `matmul` (tiling for data reuse — the big perf lesson). Copy this notebook, swap the `.cu` body, keep the compile/test/bench cells. To run the *whole engine* on this T4, push the repo to GitHub and `!git clone` it here, then `!python bench/run.py` for GPU throughput numbers to put in `BENCHMARKS.md`.